# 02 · Agentic retrieval

A retrieval plan written before reading cannot always name the entity needed
for a later search. Agent tools allow a different workflow: search for an initial
fact, read its source, and use what was found to choose the next query. This
adds flexibility, along with opportunities to waste searches or miss evidence.

The examples here expose the search, read, and reflection tools directly, so
their state changes are visible without ADK or a model service. Recorded MuSiQue
trajectories show where an actual agent failed despite having those tools.
Search-budget and citation experiments help separate tool access, evidence
availability, and factual correctness.

## 1. Search, reading, and evidence

The decomposition arm plans all retrieval queries before reading. Here an agent
can search, read a discovered entity, and choose the next query from that evidence.
The tools are ordinary annotated Python functions; ADK wraps them only in live
execution. Each question gets fresh tools, session, and usage scope.

| Tool | Purpose | Constraint |
| --- | --- | --- |
| `plan_retrieval` | Record the evidence needed | Cannot reset after search |
| `search_musique` | Dense search with local embeddings | Shares the search budget |
| `grep_musique` | Literal phrase search | Shares the same budget |
| `read_musique` | Read full passages behind previews | Only IDs exposed by search |
| `reflect_evidence` | Record remaining evidence gaps | Read before reflecting |

Search previews locate evidence; they are not permission to cite. The recorder
tracks exposed and read IDs independently of the answer JSON. Three search
operations and ten model calls bound the live loop. `not_stated` due to missing
evidence and `model_call_limit` due to a guard are distinct recorded outcomes.

### Tool access and policy are separate

`plan → search previews → read source IDs → update evidence gaps → choose another search → answer or abstain`

The manual policy below intentionally takes the useful route. In a live run ADK
chooses the tool calls, and may choose poorly. Search budgets count dense and
literal searches together. The saved run uses three searches and ten model
calls; those limits do not imply every question needs that many operations.

In [1]:
# ruff: noqa: E402 - resolve checkout imports before the first example
import json
import sys
from pathlib import Path

from IPython.display import Markdown
from IPython.display import display as show


# True runs each model step on MODELS through the proxy and caches the result.
# False replays the reference results from RI #2's release with no model calls.
RUN_EXPENSIVE = False

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "aieng-advanced-rag/aieng").is_dir())
sys.path[:0] = [str(ROOT / "aieng-advanced-rag"), str(ROOT / "implementations/cross_document_synthesis")]
from synthesis_lab.tutorial import require_compute


require_compute()

from aieng.advanced_rag.models import choose_models


MODELS = choose_models(show=True)

import numpy as np
from aieng.advanced_rag.corpora.musique import Passage
from aieng.advanced_rag.evaluation.multihop import exact_match
from synthesis_lab.agent_lab import MusiqueTools, parse_lab_answer
from synthesis_lab.reference import agent_lab


# The run holds MuSiQue question text and agent traces, so it ships in RI #2's
# reference release.
lab = agent_lab(models=MODELS, live=RUN_EXPENSIVE)
report = lab.value

configuration = {
    key: report[key]
    for key in (
        "framework",
        "adk_version",
        "model",
        "base_url",
        "embedding_model",
        "embedding_revision",
        "corpus_revision",
        "corpus_passages",
        "chat_template_kwargs",
        "max_searches",
        "max_model_calls",
    )
}
print(json.dumps(configuration, indent=2))
assert configuration["base_url"] == "https://proxy.vectorinstitute.ai/v1"
assert configuration["chat_template_kwargs"] == {"enable_thinking": False}

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer)
agent-lab reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
          27 calls, 52,782 prompt + 1,128 completion tokens
          rebuilding with gemini-3.5-flash-lite: about 27 calls, est. $0.02 (set RUN_EXPENSIVE = True to rebuild)
{
  "framework": "ADK",
  "adk_version": "2.10.0",
  "model": "Qwen3_6-35B-A3B",
  "base_url": "https://proxy.vectorinstitute.ai/v1",
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "embedding_revision": "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a",
  "corpus_revision": "5ec05b38deecc3318bb432c69865959c56058990",
  "corpus_passages": 11656,
  "chat_template_kwargs": {
    "enable_thinking": false
 

## 2. A worked example

Reuse the composer → birthplace → country chain from notebook 01. The query
encoder below is explicitly scripted for three queries. We call the real tools
manually so you can inspect state changes without a framework or a model.
Read the composer before substituting the name into the birthplace query, then
read the town before forming the country query. Gold MuSiQue data is absent.

In [2]:
toy_passages = [
    Passage("toy::0", "Composer", "Mira Vale composed The Glass Harbor."),
    Passage("toy::1", "Other opera", "The Golden Shore is an unrelated opera."),
    Passage("toy::2", "Other place", "Southmere is a different country."),
    Passage("toy::3", "Other composer", "Tara Reed composed a different opera."),
    Passage("toy::4", "Other town", "Birchwell is unrelated to this question."),
    Passage("toy::5", "Birthplace", "Mira Vale was born in Alderwick."),
    Passage("toy::6", "Country", "Alderwick is a town in Northmere."),
    Passage("toy::7", "Premiere", "The Glass Harbor premiered in 1890."),
]
toy_vectors = np.eye(len(toy_passages), dtype=np.float32)
toy_queries = ["Who composed The Glass Harbor?", "Where was Mira Vale born?", "Which country contains Alderwick?"]
toy_rows = dict(zip(toy_queries, [0, 5, 6], strict=True))


def toy_encode(queries):
    """Return scripted vectors for the fixture, not learned embeddings."""
    return [toy_vectors[toy_rows[query]].tolist() for query in queries]


tools = MusiqueTools(toy_passages, toy_vectors, toy_encode, k=1, max_searches=3)
tools.plan_retrieval(["composer", "birthplace", "country"])
for query, gap in zip(toy_queries, ["birthplace", "country", ""], strict=True):
    hits = tools.search_musique(query)
    ids = [hit["passage_id"] for hit in hits["hits"]]
    print("Query:", query, "| Preview IDs:", ids)
    print("Read:", tools.read_musique(ids)["passages"])
    reflection = tools.reflect_evidence([gap] if gap else [])
    print("Remaining searches:", reflection["searches_remaining"])
toy_answer = parse_lab_answer(
    json.dumps({"status": "answered", "answer": "Northmere", "citations": ["toy::0", "toy::5", "toy::6"]}), tools
)
print("Answer with validated source IDs:", toy_answer)
assert set(toy_answer["citations"]) <= tools.read_ids <= tools.exposed_ids

Query: Who composed The Glass Harbor? | Preview IDs: ['toy::0']
Read: [{'passage_id': 'toy::0', 'text': 'Composer\nMira Vale composed The Glass Harbor.'}]
Remaining searches: 2
Query: Where was Mira Vale born? | Preview IDs: ['toy::5']
Read: [{'passage_id': 'toy::5', 'text': 'Birthplace\nMira Vale was born in Alderwick.'}]
Remaining searches: 1
Query: Which country contains Alderwick? | Preview IDs: ['toy::6']
Read: [{'passage_id': 'toy::6', 'text': 'Country\nAlderwick is a town in Northmere.'}]
Remaining searches: 0
Answer with validated source IDs: {'status': 'answered', 'answer': 'Northmere', 'citations': ['toy::0', 'toy::5', 'toy::6']}


### Inspect the provenance boundary

**Predict first:** does a known passage ID suffice for a citation if the agent
never read it? Then ask a different question: can a factually wrong answer still
pass structural citation validation?

In [3]:
try:
    parse_lab_answer(json.dumps({"status": "answered", "answer": "Northmere", "citations": ["toy::7"]}), tools)
except ValueError as error:
    print("Unread citation rejected:", error)

wrong_but_well_formed = parse_lab_answer(
    json.dumps({"status": "answered", "answer": "Southmere", "citations": ["toy::0", "toy::5", "toy::6"]}), tools
)
print("Structure accepted:", wrong_but_well_formed)
print("Scorer-only answer EM:", exact_match(wrong_but_well_formed["answer"], ["Northmere"]))

Unread citation rejected: citations must uniquely identify passages actually read
Structure accepted: {'status': 'answered', 'answer': 'Southmere', 'citations': ['toy::0', 'toy::5', 'toy::6']}
Scorer-only answer EM: 0.0


The parser rejects unread citations and malformed output. It does not prove the
answer follows from the passages. Answer metrics or a separate grounding judge
address another question. Keep this distinction when interpreting a successful
tool trajectory or a high valid-citation rate.

## 3. Recorded results

The saved run used the first MuSiQue question at each hop count. All three
abstained, and one reached the model-call limit. Start with those facts rather
than interpreting more tool calls as better reasoning. The model never read all
gold support passages in these cases. Valid citations establish availability,
not entailment or correctness.

In [4]:
rows = [
    "| Hops | Status | Stop reason | Searches | Model calls | Tokens | All supports read |",
    "| --- | --- | --- | ---: | ---: | ---: | --- |",
]
for row in report["results"]:
    rows.append(
        f"| {row['hops']} | {row['status']} | {row['stop_reason']} | {row['searches']} | "
        f"{row['model_calls']} | {row['usage']['total_tokens']} | {row['all_supports_read']} |"
    )
show(Markdown(chr(10).join(rows)))
assert len(report["results"]) == 3
for row in report["results"]:
    assert row["searches"] <= report["max_searches"]
    assert row["model_calls"] <= report["max_model_calls"]
    assert set(row["citations"]) <= set(row["read_ids"]) <= set(row["exposed_ids"])
    assert row["usage"]["calls_missing_usage"] == 0
print("Total tokens:", sum(row["usage"]["total_tokens"] for row in report["results"]))

| Hops | Status | Stop reason | Searches | Model calls | Tokens | All supports read |
| --- | --- | --- | ---: | ---: | ---: | --- |
| 2 | not_stated | completed | 3 | 8 | 15495 | False |
| 3 | not_stated | model_call_limit | 3 | 10 | 18026 | False |
| 4 | not_stated | completed | 3 | 9 | 20389 | False |

Total tokens: 53910


### Inspect one trajectory

Change `EXAMPLE_INDEX` to 0, 1, or 2. Follow queries in order: did the agent read
useful evidence before spending the next search? Did a long literal phrase return
no hits? The three-hop case stopped at the model guard; its spent usage remains
recorded. ADK service timing includes work between responses and is approximate.

In [5]:
EXAMPLE_INDEX = 0
example = report["results"][EXAMPLE_INDEX]
for index, step in enumerate(example["trace"], start=1):
    details = {key: value for key, value in step.items() if key != "tool"}
    print(index, step["tool"], json.dumps(details))
print("Read IDs:", example["read_ids"])
print("Final:", {key: example[key] for key in ("status", "answer", "citations", "stop_reason")})

1 plan_retrieval {"needs": ["Who is the person Messi's Copa del Rey goals were compared to?", "When was that person signed by Barcelona?"]}
2 search_musique {"query": "Messi Copa del Rey goals compared to", "passage_ids": ["musique::00019", "musique::00016", "musique::00009", "musique::00007", "musique::00001"], "search_number": 1}
3 grep_musique {"query": "Messi Copa del Rey goals compared", "passage_ids": [], "search_number": 2}
4 grep_musique {"query": "Copa del Rey goals compared to", "passage_ids": [], "search_number": 3}
5 read_musique {"passage_ids": ["musique::00019", "musique::00016", "musique::00009", "musique::00007", "musique::00001"]}
6 reflect_evidence {"missing": ["The specific person Messi's Copa del Rey goals were compared to is not explicitly stated in the provided passages.", "The date when that person was signed by Barcelona is not provided."]}
Read IDs: ['musique::00001', 'musique::00007', 'musique::00009', 'musique::00016', 'musique::00019']
Final: {'status': 'not

### Diagnose before changing the prompt

| Observation in a trace | Candidate failure | Next check |
| --- | --- | --- |
| No useful ID appears in hits | Query/retrieval | Try an evidence-derived entity; compare literal and dense search |
| Useful preview appears but is never read | Tool policy | Inspect read calls before the next search |
| Source is read but the next query ignores it | Evidence update | Identify the missing relation explicitly |
| Needed sources are read but the answer is wrong | Synthesis | Inspect text support, not just citation IDs |
| Model-call guard fires | Execution budget | Check wasted calls; retain the spent usage |

The three recorded questions all abstained. They demonstrate the lab's migration
to local embeddings and the Vector proxy, not a measured agent improvement.

## 4. Experiments

Change `SEARCH_BUDGET` from two to three. **Predict first:** which fact will be
missing with two searches, and can the remaining evidence justify the country?
The cell creates fresh tools each time, so rerunning cannot inherit earlier reads.
A bounded loop should surface the missing fact, not silently widen its budget.

In [6]:
SEARCH_BUDGET = 2
experiment = MusiqueTools(toy_passages, toy_vectors, toy_encode, k=1, max_searches=SEARCH_BUDGET)
experiment.plan_retrieval(["composer", "birthplace", "country"])
for query in toy_queries:
    hits = experiment.search_musique(query)
    if "error" in hits:
        print("Stopped:", hits["error"])
        break
    experiment.read_musique([hit["passage_id"] for hit in hits["hits"]])
print("Searches spent:", experiment.searches, "| Read IDs:", sorted(experiment.read_ids))
print("Missing synthetic supports:", sorted({"toy::0", "toy::5", "toy::6"} - experiment.read_ids))
try:
    parse_lab_answer(json.dumps({"status": "answered", "answer": "Northmere", "citations": ["toy::7"]}), experiment)
except ValueError as error:
    print("Unread-citation guard:", str(error))

Stopped: Search budget exhausted; read existing hits and finish.
Searches spent: 2 | Read IDs: ['toy::0', 'toy::5']
Missing synthetic supports: ['toy::6']
Unread-citation guard: citations must uniquely identify passages actually read


### Compare short and long literal searches

Edit `PHRASES` while leaving the budget fixed. Literal search does not paraphrase:
an over-specific phrase can spend a round without finding a passage. Dense and
literal searches share the same counter. This is a useful diagnosis for the saved
trajectory, not proof that one search method always wins.

In [7]:
PHRASES = ["Mira Vale", "When was the composer of The Glass Harbor born in Alderwick?"]
lexical = MusiqueTools(toy_passages, toy_vectors, toy_encode, k=5, max_searches=3)
lexical.plan_retrieval(["composer and birthplace"])
for phrase in PHRASES:
    hits = lexical.grep_musique(phrase)
    print(
        phrase,
        "->",
        [hit["passage_id"] for hit in hits.get("hits", [])],
        "| Searches remaining:",
        hits.get("searches_remaining"),
    )

Mira Vale -> ['toy::0', 'toy::5'] | Searches remaining: 2
When was the composer of The Glass Harbor born in Alderwick? -> [] | Searches remaining: 1


**Expected observation:** with two searches the country passage remains unread;
a shorter literal phrase can find an entity while a long question-shaped phrase
finds nothing. Increasing a budget does not improve the model's search policy
by itself. Record what changed and rerun with a fresh tool instance.

## 5. Run the agent on your model

The recorded run in section 3 comes from `agent_lab`, one step behind
`load_or_run`. With `RUN_EXPENSIVE = False` it replays the reference built with
`Qwen3_6-35B-A3B` from RI #2's release. With `RUN_EXPENSIVE = True` it asks the
same three questions on `MODELS`, with fresh sessions and meters, and caches the
result. It loads the pinned corpus, the local BGE model, and the ADK runtime. A
live run can cost money even when it abstains or reaches a guard. Supply
credentials through onboarding (`OPENAI_API_KEY` plus the Vector `OPENAI_BASE_URL`)
or `VECTOR_PROXY_API_KEY` in the environment or repository `.env`, never in a
notebook cell.

Live chat uses the shared model roles and Vector credential resolver. The
default family is Google; use `ADVANCED_RAG_MODEL_FAMILY=onprem` for Qwen
generation and extraction. Embeddings stay local. Set
`ADVANCED_RAG_SPEND_LIMIT_USD` before execution to limit spending. See the
[provider settings](GUIDE.md#models-credentials-and-spend). Recorded results
retain their original models; fresh comparisons require matched model roles.

In [8]:
# The first code cell ran the agent on MODELS when RUN_EXPENSIVE was True and
# replayed the reference otherwise. The label says which.
print(lab.label)

agent-lab reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
          27 calls, 52,782 prompt + 1,128 completion tokens
          rebuilding with gemini-3.5-flash-lite: about 27 calls, est. $0.02 (set RUN_EXPENSIVE = True to rebuild)


## 6. Interpretation

The hand-run path succeeds because its policy chooses the useful queries. The
recorded model path shows that tools alone do not guarantee useful behavior.
Do not compare this three-question sample with notebook 01's balanced sample as
an A/B test: their selections and budgets differ.

Before moving on:

1. Identify one avoidable search in a recorded trajectory and propose a revision.
2. Explain why preview access, reading, citation validity, and entailment differ.
3. Specify the matched IDs and evidence/model budgets needed for a fair comparison.

[Notebook 03](03_ircot.ipynb) adds tool-enforced retrieval/update alternation and
shows the scored IRCoT arm. [Notebook 05](05_decision_grid.ipynb) compares complete
cohorts and the measured optional corrective policy.

Keep gold answers and hop graphs with the scorer. Local pinned embeddings and
the Vector proxy are used for live RI #2 experiments. Source IDs establish
provenance; they do not by themselves establish factual entailment.

See the [participant guide](GUIDE.md), [corpus/provenance record](CORPUS.md),
[experiment contracts](docs/EXPERIMENTS.md), and [full matched comparison](docs/COMPARISONS.md).